### Лабораторная работа №3

#### Импорты

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.graph_objects as go
from ipywidgets import widgets

#### Набор данных «Ирисы Фишера»

##### Получение и рассмотренеи данных

In [2]:
iris_dict: dict = load_iris()
iris_dict.keys()

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])

In [3]:
iris_df = pd.DataFrame(data=iris_dict['data'], columns=iris_dict['feature_names'])
iris_df['target'] = iris_dict['target']

In [4]:
iris_df.sample(5)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
13,4.3,3.0,1.1,0.1,0
28,5.2,3.4,1.4,0.2,0
18,5.7,3.8,1.7,0.3,0
104,6.5,3.0,5.8,2.2,2
47,4.6,3.2,1.4,0.2,0


In [5]:
iris_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


In [6]:
iris_df.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


##### Преобразование данных

In [7]:
y_iris = iris_df['target']
X_iris = iris_df.drop(columns=['target'])

In [8]:
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

In [9]:
scaler_class = StandardScaler()
X_train_iris_scaled = scaler_class.fit_transform(X_train_iris)
X_test_iris_scaled = scaler_class.transform(X_test_iris)

In [10]:
scaler_сlust = StandardScaler()
X_iris_scaled = scaler_сlust.fit_transform(X_iris)

##### Классификация набора алгоритмом k-ближайших соседей

##### Классификация набора алгоритмом «случайный лес»

##### Классификация набора машинами опорных векторов (SVM)

##### Кластеризация набора алгоритмом k-средних

Применение алгоритма

In [24]:
inertia = []
cluster_centers_list = []
labels_list = []
n_clusters_list = [n_clusters for n_clusters in range(1, 11)]
for n_clusters in n_clusters_list:
    kmeans_сlust_model = KMeans(n_clusters=n_clusters)
    kmeans_сlust_model.fit(X_iris_scaled)
    inertia.append(kmeans_сlust_model.inertia_)
    cluster_centers_list.append(kmeans_сlust_model.cluster_centers_)
    labels_list.append(kmeans_сlust_model.labels_)


Визуализация распределения

In [42]:
# https://plotly.com/python/figurewidget-app/

feature_names_iris = list(X_iris.columns)

n_clusters_slider = widgets.IntSlider(
    value=n_clusters_list[0],
    min=n_clusters_list[0],
    max=n_clusters_list[-1],
    step=1.0,
    description='Количество кластеров:',
)

show_cluster_centers = widgets.Checkbox(
    description='Показывать центр кластеров: ',
    value=True,
)

container = widgets.HBox(children=[show_cluster_centers, n_clusters_slider])

y_feature = widgets.Dropdown(
    description='Признак Y:   ',
    value=feature_names_iris[0],
    options=feature_names_iris
)

x_feature = widgets.Dropdown(
    description='Признак X:   ',
    value=feature_names_iris[1],
    options=feature_names_iris
)

In [72]:
g = go.FigureWidget(layout=dict(width=1200, height=500))
g.add_scatter(
    mode='markers',
    marker=dict(colorscale='Viridis', showscale=False, size=10, opacity=0.75),
    name='Точка',
    hovertemplate='Точка кластера: %{text}<br>X: %{x}<br>Y: %{y}<extra></extra>'
)
g.add_scatter(
    mode='markers',
    marker=dict(symbol='x', size=12, color='red'),
    name='Центр кластера',
    hovertemplate='Центр кластера: %{text}<br>X: %{x}<br>Y: %{y}<extra></extra>'
)

def response(change=None):
    # Получаем текущие значения виджетов
    x_name = x_feature.value
    y_name = y_feature.value
    x_idx = feature_names_iris.index(x_name)
    y_idx = feature_names_iris.index(y_name)
    n = n_clusters_slider.value
    show = show_cluster_centers.value

    # Индекс для выбранного k в списке n_clusters_list
    idx = n_clusters_list.index(n)

    # Берём метки кластеров и центры
    labels = labels_list[idx].tolist()
    centers = cluster_centers_list[idx]

    # Координаты точек по выбранным признакам
    x_vals = X_iris_scaled[:, x_idx].tolist()
    y_vals = X_iris_scaled[:, y_idx].tolist()

    # Координаты центров по выбранным признакам
    x_centers = centers[:, feature_names_iris.index(x_name)].tolist()
    y_centers = centers[:, feature_names_iris.index(y_name)].tolist()

    # Обновляем график (batch_update для производительности)
    with g.batch_update():
        # Точки
        g.data[0].x = x_vals
        g.data[0].y = y_vals
        g.data[0].marker.color = labels
        g.data[0].text = labels
        # Центры
        g.data[1].x = x_centers
        g.data[1].y = y_centers
        g.data[1].text = [str(i) for i in range(n)]
        g.data[1].visible = show
        # Подписи осей и заголовок
        g.layout.xaxis.title = x_name
        g.layout.yaxis.title = y_name
        g.layout.title = f'Кластеризация набора "Ирисы Фишера" (n_clusters={n})'

x_feature.observe(response, names='value')
y_feature.observe(response, names='value')
n_clusters_slider.observe(response, names='value')
show_cluster_centers.observe(response, names='value')

response()

container = widgets.HBox([show_cluster_centers, n_clusters_slider])
container2 = widgets.HBox([x_feature, y_feature])
widgets.VBox([container, container2, g])

Визуализация метода локтя

In [62]:
fig_kmeans_сlust_elbow_method = go.Figure()
fig_kmeans_сlust_elbow_method.add_trace(
    go.Scatter(x=n_clusters_list, y=inertia, mode='lines+markers', line=dict(color='red', width=2), marker=dict(color='red', size=10))
)
fig_kmeans_сlust_elbow_method.add_trace(
    go.Bar(x=n_clusters_list, y=inertia, marker_color='blue')
)
fig_kmeans_сlust_elbow_method.update_layout(
    title='Метод локтя "Ирисы Фишера"',
    xaxis_title='Число кластеров (n_clusters)',
    yaxis_title='Сумма внутрикластерных расстояний',
    xaxis=dict(tickmode='linear', dtick=1),
    barcornerradius=20,
    showlegend=False
)
fig_kmeans_сlust_elbow_method.show()

##### Иерархическая кластеризация методом Уорда